# Bonus 01 — Tools over Real Tables (DuckDB)
**Optional | After Lab 1B | Colab CPU | OpenAI API key**

Lab 1B taught the two-round tool loop on a toy SQLite table. This bonus keeps that loop and adds what production agents actually hit: **a real file-backed table** (DuckDB + CSV) and **more than one tool in the same turn**.

You already know weather-from-Open-Meteo. Here it is one tool among three, next to flight search and a city-fact lookup.

> **The key idea:** The model still never opens the CSV. You expose `get_flight` / `get_fact` / `get_weather`. Parameterized SQL stays in Python.

```
user question
  → model may request get_flight AND get_fact in one turn
  → your code runs DuckDB / HTTP
  → role=tool observations
  → grounded answer
```


In [ ]:
import sys
%pip install -q uv
!uv pip install -q --python {sys.executable} openai duckdb httpx pandas python-dotenv

In [ ]:
import os
try:
    from google.colab import userdata          # Colab: read the Secret you added
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv             # local: read .env in the repo root
    load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Add OPENAI_API_KEY as a Colab Secret or to .env"

OPENAI_API_KEY  = os.environ["OPENAI_API_KEY"]
OPENAI_BASE_URL = "https://api.openai.com/v1"
DEFAULT_MODEL   = "gpt-4o-mini"

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
print(f"Ready — {DEFAULT_MODEL} at {OPENAI_BASE_URL}")

## 1. Put the CSVs where DuckDB can see them

On Colab the notebook is not next to `Bonus/`. We download the course CSVs if they are missing, then peek at the tables.


In [ ]:
from pathlib import Path
import urllib.request
import duckdb
import pandas as pd

RAW = "https://raw.githubusercontent.com/tatwan/mastering_llm_deployments/main/Bonus"

def ensure(name: str):
    if Path(name).exists():
        return
    urllib.request.urlretrieve(f"{RAW}/{name}", name)
    print("downloaded", name)

ensure("flight_data.csv")
ensure("fun_facts.csv")

conn = duckdb.connect("city_tour.db")
conn.execute("CREATE OR REPLACE TABLE flight AS SELECT * FROM 'flight_data.csv'")
conn.execute("CREATE OR REPLACE TABLE fun_facts AS SELECT * FROM 'fun_facts.csv'")
print(conn.execute("SHOW TABLES").fetchall())
print()
print("Flights (first 5)")
print(conn.execute("SELECT * FROM flight LIMIT 5").df())
print()
print("Facts (first 5)")
print(conn.execute("SELECT * FROM fun_facts LIMIT 5").df())


**Checkpoint:** you should see cities like New York, Dubai, Amman. If download failed, clone the repo and run this notebook from `Bonus/`.

## 2. Three small tools

**Why parameterized SQL:** the model supplies city names as *data*. String-concatenated SQL is how agents become injection bugs (Lab 2 / Lab 11).


In [ ]:
import json
import httpx

def get_weather(latitude: float, longitude: float) -> str:
    try:
        r = httpx.get("https://api.open-meteo.com/v1/forecast",
                      params={"latitude": latitude, "longitude": longitude, "current_weather": True}, timeout=10)
        r.raise_for_status()
        return r.text
    except Exception as e:
        return json.dumps({"error": type(e).__name__, "mock": True, "temperature": 22.0})

print(get_weather(31.95, 35.93)[:120])

The two table tools share one shape: a parameterized `SELECT`, rows turned into a list of dicts, and a JSON error instead of an exception when nothing matches. The model reads that error and explains it instead of crashing the loop.

In [ ]:
def rows_as_json(rows) -> str:
    cols = [d[0] for d in conn.description]
    return json.dumps([dict(zip(cols, row)) for row in rows])

def get_flight(from_city: str, to_city: str) -> str:
    rows = conn.execute("SELECT * FROM flight WHERE lower(from_city)=lower(?) AND lower(to_city)=lower(?)",
                        [from_city, to_city]).fetchall()
    return rows_as_json(rows) if rows else json.dumps({"error": "No flights found", "from_city": from_city, "to_city": to_city})

def get_fact(city: str) -> str:
    rows = conn.execute("SELECT * FROM fun_facts WHERE lower(City)=lower(?)", [city]).fetchall()
    return rows_as_json(rows) if rows else json.dumps({"error": "No facts found", "city": city})

print(get_flight("Paris", "Rome"))
print(get_fact("Amman"))

## 3. Schemas the model can see

Same contract as Lab 1B: name, description, JSON arguments. Three tools this time.


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Current weather for a latitude and longitude (Open-Meteo).",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {"type": "number"},
                    "longitude": {"type": "number"},
                },
                "required": ["latitude", "longitude"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_flight",
            "description": "Look up a flight row between two cities in the local table.",
            "parameters": {
                "type": "object",
                "properties": {
                    "from_city": {"type": "string"},
                    "to_city": {"type": "string"},
                },
                "required": ["from_city", "to_city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_fact",
            "description": "Look up a fun fact for a city in the local table.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            },
        },
    },
]
DISPATCH = {"get_weather": get_weather, "get_flight": get_flight, "get_fact": get_fact}
print(len(tools), "tools ready")


## 4. One loop, N tools

If the model requests two tools in one assistant turn, send **one** `role=tool` message per `tool_call_id` before round 2. That is parallel tool use — the efficiency feature Lab 1B mentioned.


In [ ]:
def run_agent(question: str) -> str:
    messages = [
        {"role": "system", "content": "Use tools for weather, flights, and city facts. Do not invent table rows."},
        {"role": "user", "content": question},
    ]
    first = client.chat.completions.create(
        model=DEFAULT_MODEL, messages=messages, tools=tools, tool_choice="auto"
    )
    msg = first.choices[0].message
    if not msg.tool_calls:
        return msg.content
    messages.append(msg)
    for call in msg.tool_calls:
        args = json.loads(call.function.arguments)
        print("Tool:", call.function.name, args)
        fn = DISPATCH.get(call.function.name)
        result = fn(**args) if fn else json.dumps({"error": "unknown tool"})
        print("Observation:", str(result)[:180])
        messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
    second = client.chat.completions.create(model=DEFAULT_MODEL, messages=messages)
    return second.choices[0].message.content

print(run_agent("Is there a flight from Paris to Rome, and what is a fun fact about Paris?"))


**Checkpoint:** you should see **two** tool calls (flight + fact) in one turn, then an answer that uses both observations.

Now read the prices in the answer. When we ran it, the model quoted them in euros. The table has a `price` column and no currency at all, so the model supplied one from nowhere. It is a small invention, and exactly the kind a tool is supposed to prevent: the tool returned the truth, and the model decorated it. Stretch goal 3 fixes it at the source.

Try a weather question next (the model must pick lat/long itself):


In [ ]:
print(run_agent("What is the weather in Amman, Jordan right now?"))


## Bonus 01 complete

- [ ] DuckDB tables loaded from CSV
- [ ] Parameterized `get_flight` / `get_fact`
- [ ] One question that triggered two tools
- [ ] Weather still works as an HTTP tool

## Stretch

1. Ask for a city with no flight row. The tool should return JSON `error`, and the model should say so.
2. Add `get_cheapest_from(city)` that runs `SELECT * FROM flight WHERE from_city=? ORDER BY price LIMIT 1`.
3. Make `get_flight` return `"currency": "unknown"` with each row, or add a system-prompt rule against inventing units. Which works better?

Next: [Bonus 02](02_react_agent.ipynb), the text ReAct loop against a real model, or [Lab 2](../02_Prompting/README.md) if you are still on the 2-day clock.
